# Cuaderno U1-04. Variables, parametros, supuestos y condiciones de frontera

**Modelacion y Simulacion Computacional** · Maestria en Ingenieria · Universidad de Sucre

Unidad 1, Fundamentos de modelacion en ingenieria · Subtema 1.4 del plan de asignatura

Docente Daniel David Otero Meza · Periodo 2026-2

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad1/U1_04_variables_parametros_y_fronteras.ipynb)

*El docente reemplaza `msc-unisucre/msc2026-material` por la direccion real del repositorio del
curso antes de publicar el cuaderno.*

Este cuaderno acompana la Seccion 1.4 del libro y toma de la Seccion 1.5 las herramientas que sirven para revisar una formulacion, que son la coherencia dimensional y el teorema Pi. Escribe la forma canonica de un modelo dinamico, separa estados de entradas, perturbaciones y parametros, levanta el registro de supuestos, cuenta grados de libertad y reproduce los Ejemplos 1.2 y 1.3 del libro.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante estara en capacidad de hacer lo siguiente.

1. Escribir un modelo dinamico en la forma canonica de la Ecuacion 1.4 y clasificar cada magnitud como estado, entrada, perturbacion o parametro.
2. Levantar el registro de supuestos de cuatro columnas y comprobar que cada supuesto tenga una prueba prevista.
3. Contar los grados de libertad de una formulacion estacionaria por el rango del jacobiano, antes de programarla.
4. Reproducir el Ejemplo 1.2 del libro, el tirante normal de un canal trapezoidal, y clasificar el regimen del flujo.
5. Distinguir las condiciones de Dirichlet, Neumann y Robin y reconocer cuando una formulacion carece de solucion unica.

## Puesta a punto

La primera celda detecta el entorno e instala unicamente lo que falte. La segunda fija la semilla del curso y la paleta del libro. La tercera define las funciones de verificacion que se usan mas abajo. Ejecutelas en orden antes de continuar.

In [ ]:
# Puesta a punto del entorno. Detecta Colab e instala solo lo que falte.
import importlib
import importlib.util
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes):
    """Instala los paquetes ausentes sin reinstalar los que ya estan."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)
    return faltantes


AUSENTES = asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                     "matplotlib": "matplotlib", "sympy": "sympy"})

print("Entorno de ejecucion:", "Google Colab" if EN_COLAB else "JupyterLab local")
print("Paquetes instalados en esta sesion:", AUSENTES or "ninguno, ya estaban")

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

# Semilla unica de la asignatura. Ningun resultado depende de una corrida.
SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

# Paleta del libro. Los cuadernos usan los mismos colores que las figuras.
PALETA = {
    "azul": "#1F4E79",
    "rojo": "#B3251E",
    "verde": "#2E7D32",
    "naranja": "#E07B00",
    "gris": "#5A5A5A",
    "morado": "#6A3D9A",
}

plt.rcParams.update({
    "figure.figsize": (8.6, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
    "font.size": 10.0,
    "legend.frameon": True,
    "legend.framealpha": 0.92,
})

print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · pandas {pd.__version__}")
print(f"SymPy {sp.__version__} · semilla del curso {SEMILLA}")

In [ ]:
# Bandera de revision de los ejercicios guiados.
# Mientras valga False el cuaderno se ejecuta completo aunque falten celdas.
# Pongala en True cuando haya completado las celdas marcadas para completar.
REVISAR = False
print("REVISAR =", REVISAR)

In [ ]:
def verificar_libro(nombre, obtenido, publicado, tolerancia=1e-3, unidad=""):
    """Contrasta un resultado calculado con la cifra que publica el libro."""
    valor = float(obtenido)
    escala = abs(publicado) if publicado else 1.0
    error = abs(valor - publicado) / escala
    print(f"{nombre:<46s} calculado {valor:>12.6g} {unidad:<10s}"
          f" libro {publicado:>12.6g}  error rel. {error:.1e}")
    assert error <= tolerancia, f"{nombre} se aparta de la cifra publicada"
    return valor


def comprobar(nombre, obtenido, referencia, tolerancia=1e-3, unidad=""):
    """Revisa una celda de ejercicio contra su valor de referencia.

    Con REVISAR en False solo informa que el ejercicio sigue pendiente, de modo
    que el cuaderno nunca se detiene por una celda sin completar.
    """
    if not REVISAR:
        print(f"[pendiente]  {nombre}")
        return False
    valor = float(obtenido)
    escala = abs(referencia) if referencia else 1.0
    error = abs(valor - referencia) / escala
    marca = "correcto " if error <= tolerancia else "revisar  "
    print(f"[{marca}]  {nombre} = {valor:.6g} {unidad}"
          f"  referencia {referencia:.6g}  error rel. {error:.2e}")
    assert error <= tolerancia, f"{nombre} no coincide con la referencia"
    return True


def ruta_datos(nombre):
    """Ubica un archivo de la carpeta datos sin usar rutas absolutas.

    Funciona igual en Colab, donde el cuaderno suele abrirse en el directorio
    de trabajo, y en una copia local del repositorio, donde el cuaderno vive
    dentro de Unidad1 o de soluciones.
    """
    candidatas = (Path("datos"),
                  Path("..") / "datos",
                  Path("..") / ".." / "datos",
                  Path("03_cuadernos") / "datos")
    for base in candidatas:
        if (base / nombre).exists():
            return base / nombre
    for base in candidatas:        # la carpeta existe pero el archivo aun no
        if base.is_dir():
            return base / nombre
    return Path("datos") / nombre  # entorno nuevo, como una sesion de Colab


def cargar_o_generar(nombre, generador):
    """Lee el archivo de datos y, si no esta, lo reconstruye con la semilla."""
    ruta = ruta_datos(nombre)
    if ruta.exists():
        print(f"Datos leidos de {ruta}")
        return pd.read_csv(ruta)
    tabla = generador()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False)
    print(f"Datos regenerados con la semilla {SEMILLA} y guardados en {ruta}")
    return tabla


def integrar_trapecio(valores, muestras):
    """Regla del trapecio compatible con NumPy 1 y con NumPy 2."""
    regla = getattr(np, "trapezoid", None) or np.trapz
    return float(regla(valores, muestras))


print("Funciones auxiliares disponibles.")

## 1. La forma canonica y el reparto de papeles

Casi todos los modelos dinamicos de la ingenieria admiten la forma canonica de
la Ecuacion 1.4 del libro, en la que el estado evoluciona segun una funcion de
si mismo, de las entradas manipulables, de las perturbaciones y de los
parametros, y la salida se obtiene del estado y de las entradas.

El Listado 1.3 del libro escribe esa separacion en codigo. La celda siguiente la
aplica a un tanque agitado con reaccion de primer orden, donde el estado es la
concentracion, la entrada manipulable es el caudal, la perturbacion es la
concentracion de ingreso y los parametros son el volumen y la constante
cinetica.

In [ ]:
from scipy.integrate import solve_ivp


def dinamica(t, x, entradas, perturbaciones, parametros):
    """Balance de un tanque agitado con reaccion de primer orden."""
    concentracion, = x                          # vector de estado, kg/m3
    caudal = entradas(t)                        # manipulable, m3/h
    entrada = perturbaciones(t)                 # no se elige, kg/m3
    volumen, constante = parametros             # m3 y 1/h
    return [caudal / volumen * (entrada - concentracion)
            - constante * concentracion]


def simular(x0, entradas, perturbaciones, parametros, horizonte):
    """Ejecuta el modelo bajo un escenario concreto."""
    return solve_ivp(dinamica, (0.0, horizonte), x0, method="LSODA",
                     args=(entradas, perturbaciones, parametros),
                     dense_output=True, rtol=1e-8, atol=1e-10)


PARAMETROS = (120.0, 0.35)      # volumen en m3, constante cinetica en 1/h
caudal_operacion = lambda t: 40.0                       # m3/h
carga_entrante = lambda t: 1.0 + 0.6 * (4.0 <= t < 6.0)  # kg/m3, un pulso

malla = np.linspace(0.0, 20.0, 400)
respuesta = simular([0.0], caudal_operacion, carga_entrante,
                    PARAMETROS, 20.0).sol(malla)[0]

MAGNITUDES = [
    ("concentracion en el tanque", "estado", "kg/m3", "la resuelve el modelo"),
    ("caudal de operacion", "entrada", "m3/h", "el ingeniero la elige"),
    ("concentracion de ingreso", "perturbacion", "kg/m3", "actua sin poder elegirse"),
    ("volumen util", "parametro", "m3", "constante durante la simulacion"),
    ("constante cinetica", "parametro", "1/h", "constante durante la simulacion"),
    ("concentracion de salida", "salida", "kg/m3", "es lo que se observa"),
]
reparto = pd.DataFrame(MAGNITUDES, columns=["magnitud", "papel", "unidad", "razon"])

figura, eje = plt.subplots()
eje.plot(malla, [carga_entrante(t) for t in malla], color=PALETA["naranja"],
         lw=1.5, ls="--", label="Perturbacion, concentracion de ingreso")
eje.plot(malla, respuesta, color=PALETA["azul"], lw=1.9,
         label="Estado, concentracion en el tanque")
eje.set_xlabel("Tiempo transcurrido (h)")
eje.set_ylabel("Concentracion (kg/m3)")
eje.legend(loc="upper right")
eje.set_title("Forma canonica de la Ecuacion 1.4, con un pulso de perturbacion")
plt.show()

reparto.set_index("magnitud")

La Definicion 1.9 del libro exige que el vector de estado sea minimal. Si se le
agregan magnitudes que son combinacion algebraica de las demas, el sistema deja
de tener solucion unica y el integrador falla o converge a resultados que
dependen de la tolerancia. El caso clasico es incluir a la vez la masa de
soluto, el volumen y la concentracion de un tanque, porque la tercera es el
cociente de las dos primeras.

La celda siguiente lo hace visible sin necesidad de integrar nada. Basta con
mirar el rango del jacobiano del sistema algebraico que vincula las tres
magnitudes.

In [ ]:
def jacobiano_numerico(residuos, punto, paso=1e-6):
    """Jacobiano por diferencias adelantadas de un sistema de residuos."""
    punto = np.asarray(punto, dtype=float)
    base = np.asarray(residuos(punto), dtype=float)
    matriz = np.empty((base.size, punto.size))
    for j in range(punto.size):
        desplazado = punto.copy()
        desplazado[j] += paso
        matriz[:, j] = (np.asarray(residuos(desplazado)) - base) / paso
    return matriz


def estado_redundante(x):
    """Masa, volumen y concentracion, con la relacion que las hace dependientes."""
    masa, volumen, concentracion = x
    return [masa - concentracion * volumen]


punto = np.array([50.0, 120.0, 50.0 / 120.0])
matriz = jacobiano_numerico(estado_redundante, punto)
rango = int(np.linalg.matrix_rank(matriz, tol=1e-8))

print(f"jacobiano de la relacion algebraica · rango {rango} sobre "
      f"{matriz.shape[1]} magnitudes")
print("Hay una relacion algebraica entre las tres, de modo que el estado "
      "minimal tiene dos componentes y no tres.")
assert rango == 1

## 2. El registro de supuestos

Un supuesto no declarado no desaparece, se vuelve inauditable. La Tabla 1.3 del
libro fija el formato de cuatro columnas, que son el enunciado, la razon por la
cual se adopta, el efecto esperado si resultara falso y la prueba con la que
podria contrastarse. La ultima columna es la que distingue un supuesto de una
excusa.

In [ ]:
SUPUESTOS = [
    {"supuesto": "flujo uniforme y permanente",
     "razon": "el tramo es prismatico y largo frente al tirante",
     "efecto si es falso": "el tirante real varia a lo largo del tramo",
     "prueba": "perfil de flujo gradualmente variado"},
    {"supuesto": "rugosidad constante en el perimetro",
     "razon": "revestimiento homogeneo de concreto",
     "efecto si es falso": "subestimacion del tirante en la zona vegetada",
     "prueba": "aforo con medicion de tirante"},
    {"supuesto": "velocidad uniforme en la seccion",
     "razon": "canal ancho y poco profundo",
     "efecto si es falso": "error en la energia especifica y en el resalto",
     "prueba": "perfil con molinete"},
    {"supuesto": "seccion estable en el tiempo",
     "razon": "canal revestido sin arrastre",
     "efecto si es falso": "cambio de la relacion entre tirante y caudal",
     "prueba": "batimetria periodica de la seccion"},
]
registro = pd.DataFrame(SUPUESTOS)

sin_prueba = registro[registro["prueba"].str.strip() == ""]
print(f"supuestos declarados · {len(registro)}")
print(f"supuestos sin prueba prevista · {len(sin_prueba)}")
assert sin_prueba.empty, "todo supuesto debe llevar una prueba prevista"
registro.set_index("supuesto")

## 3. Grados de libertad, una comprobacion de treinta segundos

La Definicion 1.10 del libro define los grados de libertad de un modelo
estacionario como la diferencia entre el numero de incognitas independientes y
el numero de ecuaciones independientes que las vinculan. Un sistema con grados
positivos no tiene solucion unica y uno con grados negativos no tiene solucion
alguna.

El Algoritmo 1.2 y el Listado 1.4 del libro cuentan los grados por el rango del
jacobiano, que detecta ademas las ecuaciones redundantes, esto es la trampa de
escribir los balances de todos los componentes de una mezcla junto con el
balance global que es su suma.

In [ ]:
def grados_libertad(residuos, punto, n_especificadas, paso=1e-6):
    """Grados de libertad y numero de ecuaciones redundantes."""
    matriz = jacobiano_numerico(residuos, punto, paso)
    rango = int(np.linalg.matrix_rank(matriz, tol=1e-8))
    redundantes = matriz.shape[0] - rango
    return punto.size - rango - n_especificadas, redundantes


def ecuaciones_canal(x):
    """Resistencia de Manning y las tres relaciones geometricas del canal."""
    caudal, tirante, ancho, talud, rugosidad, pendiente, area, perimetro, radio = x
    return [
        caudal - area * radio**(2 / 3) * np.sqrt(pendiente) / rugosidad,
        area - (ancho + talud * tirante) * tirante,
        perimetro - (ancho + 2 * tirante * np.sqrt(1 + talud**2)),
        radio - area / perimetro,
    ]


punto_canal = np.array([12.0, 1.888, 2.50, 1.5, 0.025, 0.0008,
                        10.066, 9.307, 1.082])
gl, redundantes = grados_libertad(ecuaciones_canal, punto_canal, n_especificadas=5)

print("incognitas             9   Q, y, b, z, n, S0, A, P, R")
print("ecuaciones             4   Manning, area, perimetro y radio hidraulico")
print("especificaciones       5   b, z, n, S0 y Q")
print(f"ecuaciones redundantes {redundantes}")
print(f"grados de libertad     {gl}")
verificar_libro("grados de libertad del canal antes de especificar",
                9 - 4, 5, 1e-9)
assert gl == 0 and redundantes == 0, "la formulacion debe quedar bien planteada"
print("\nLa formulacion queda bien planteada y deja el tirante como unica "
      "incognita efectiva, tal como reporta el Ejemplo 1.2.")

### Un sistema mal planteado

Para ver que el conteo sirve de algo, conviene mirar que ocurre cuando se agrega
una ecuacion que no es independiente. Si al sistema anterior se le suma la
relacion de la velocidad media junto con la definicion del caudal como area por
velocidad, el rango no aumenta en dos sino en uno.

In [ ]:
def ecuaciones_redundantes(x):
    """El sistema del canal con una ecuacion escrita dos veces de otra forma."""
    base = ecuaciones_canal(x)
    caudal, _, _, _, _, _, area, _, radio = x
    return base + [caudal - area * radio**(2 / 3) * np.sqrt(x[5]) / x[4]]


gl_mal, redundantes_mal = grados_libertad(ecuaciones_redundantes, punto_canal, 5)
print(f"con la ecuacion repetida · redundantes {redundantes_mal} · "
      f"grados de libertad {gl_mal}")
assert redundantes_mal == 1
print("El conteo detecta la repeticion y evita creer que hay una condicion mas.")

## 4. Ejemplo 1.2 del libro, tirante normal de un canal trapezoidal

Un canal de riego de seccion trapezoidal, revestido en concreto, tiene ancho de
fondo 2.50 m, taludes con relacion horizontal a vertical de 1.5, pendiente
longitudinal 0.0008 y coeficiente de rugosidad de Manning 0.025, y debe conducir
un caudal de diseno de 12 m3/s.

El libro reporta un tirante normal de 1.888 m, area mojada de 10.066 m2,
perimetro mojado de 9.307 m, radio hidraulico de 1.082 m, velocidad media de
1.192 m/s, numero de Froude de 0.343 y tirante critico de 1.067 m, de modo que
el flujo es subcritico.

In [ ]:
from scipy.optimize import brentq

GRAVEDAD = 9.81      # m/s2
ANCHO = 2.50         # m
TALUD = 1.5          # adimensional, horizontal a vertical
PENDIENTE = 0.0008   # adimensional
RUGOSIDAD = 0.025    # s/m^(1/3)
CAUDAL_DISENO = 12.0  # m3/s


def area_mojada(tirante, ancho=ANCHO, talud=TALUD):
    """Area de la seccion trapezoidal, en m2."""
    return (ancho + talud * tirante) * tirante


def perimetro_mojado(tirante, ancho=ANCHO, talud=TALUD):
    """Perimetro mojado de la seccion trapezoidal, en m."""
    return ancho + 2.0 * tirante * np.sqrt(1.0 + talud**2)


def tirante_normal(caudal, ancho, talud, rugosidad, pendiente):
    """Tirante de flujo uniforme por busqueda de raiz, en m."""
    def residuo(y):
        area = area_mojada(y, ancho, talud)
        radio = area / perimetro_mojado(y, ancho, talud)
        return area * radio**(2 / 3) * np.sqrt(pendiente) / rugosidad - caudal
    return brentq(residuo, 1e-4, 10.0, xtol=1e-12)


def froude(caudal, tirante, ancho=ANCHO, talud=TALUD):
    """Numero de Froude de la seccion, adimensional."""
    area = area_mojada(tirante, ancho, talud)
    espejo = ancho + 2.0 * talud * tirante
    return caudal / area / np.sqrt(GRAVEDAD * area / espejo)


def tirante_critico(caudal, ancho=ANCHO, talud=TALUD):
    """Tirante que anula la energia especifica minima, en m."""
    return brentq(lambda y: froude(caudal, y, ancho, talud) - 1.0, 1e-4, 10.0,
                  xtol=1e-12)


y_normal = tirante_normal(CAUDAL_DISENO, ANCHO, TALUD, RUGOSIDAD, PENDIENTE)
area = area_mojada(y_normal)
perimetro = perimetro_mojado(y_normal)
radio = area / perimetro
velocidad = CAUDAL_DISENO / area
numero_froude = froude(CAUDAL_DISENO, y_normal)
y_critico = tirante_critico(CAUDAL_DISENO)

verificar_libro("tirante normal", y_normal, 1.888, 1e-3, "m")
verificar_libro("area mojada", area, 10.066, 1e-3, "m2")
verificar_libro("perimetro mojado", perimetro, 9.307, 1e-3, "m")
verificar_libro("radio hidraulico", radio, 1.082, 1e-3, "m")
verificar_libro("velocidad media", velocidad, 1.192, 1e-3, "m/s")
verificar_libro("numero de Froude", numero_froude, 0.343, 1e-3)
verificar_libro("tirante critico", y_critico, 1.067, 1e-3, "m")
print(f"\nregimen · {'subcritico' if numero_froude < 1 else 'supercritico'}, "
      f"con y_n mayor que y_c, luego el canal es de pendiente suave y admite "
      "control aguas abajo.")

### La banda de incertidumbre del coeficiente de rugosidad

El libro somete el resultado a una prueba de sensibilidad. Al variar el
coeficiente de rugosidad entre 0.022 y 0.030, banda habitual para concreto en
servicio, el tirante recorre el intervalo de 1.774 m a 2.061 m, esto es que una
variacion del 32 por ciento en la rugosidad produce una del 15 por ciento en el
tirante. El borde libre habitual de 0.30 m apenas cubre esa banda, de modo que
reportar el tirante como 1.888 m sin acompanarlo de ella transmite una precision
que el modelo no posee.

In [ ]:
y_baja = tirante_normal(CAUDAL_DISENO, ANCHO, TALUD, 0.022, PENDIENTE)
y_alta = tirante_normal(CAUDAL_DISENO, ANCHO, TALUD, 0.030, PENDIENTE)
variacion_rugosidad = 100.0 * (0.030 - 0.022) / RUGOSIDAD
variacion_tirante = 100.0 * (y_alta - y_baja) / y_normal

verificar_libro("tirante con rugosidad 0.022", y_baja, 1.774, 1e-3, "m")
verificar_libro("tirante con rugosidad 0.030", y_alta, 2.061, 1e-3, "m")
verificar_libro("variacion del coeficiente", variacion_rugosidad, 32.0, 1e-3, "%")
verificar_libro("variacion del tirante", variacion_tirante, 15.0, 2e-2, "%")

rugosidades = np.linspace(0.020, 0.033, 60)
tirantes = np.array([tirante_normal(CAUDAL_DISENO, ANCHO, TALUD, n, PENDIENTE)
                     for n in rugosidades])

figura, eje = plt.subplots()
eje.plot(rugosidades, tirantes, color=PALETA["azul"], lw=1.9)
eje.axvspan(0.022, 0.030, color=PALETA["naranja"], alpha=0.15, lw=0,
            label="Banda habitual para concreto en servicio")
eje.plot([RUGOSIDAD], [y_normal], "o", color=PALETA["rojo"], ms=7, zorder=5,
         label=f"Diseno, y_n = {y_normal:.3f} m")
eje.axhline(y_critico, color=PALETA["verde"], lw=1.2, ls="--",
            label=f"Tirante critico, {y_critico:.3f} m")
eje.annotate(f"{y_baja:.3f} m", (0.022, y_baja), textcoords="offset points",
             xytext=(6, -14), color=PALETA["gris"], fontsize=9)
eje.annotate(f"{y_alta:.3f} m", (0.030, y_alta), textcoords="offset points",
             xytext=(-42, 6), color=PALETA["gris"], fontsize=9)
eje.set_xlabel("Coeficiente de rugosidad de Manning (s/m^(1/3))")
eje.set_ylabel("Tirante normal (m)")
eje.legend(loc="upper left", fontsize=8.8)
eje.set_title("Ejemplo 1.2 del libro, sensibilidad del tirante a la rugosidad")
plt.show()

print(f"El borde libre habitual de 0.30 m cubre {0.30:.2f} m frente a una banda "
      f"de {y_alta - y_baja:.3f} m de ancho.")

## 5. Condiciones de frontera

La Definicion 1.11 del libro llama condicion de frontera a la relacion que la
solucion de una ecuacion en derivadas parciales debe satisfacer sobre el
contorno del dominio. Es de Dirichlet cuando prescribe el valor de la variable,
de Neumann cuando prescribe su derivada normal y de Robin cuando impone una
combinacion lineal de ambas.

El significado fisico es directo. Dirichlet corresponde a un nivel impuesto por
un elemento externo de capacidad muy grande, Neumann a un flujo impuesto, cuyo
caso homogeneo describe una pared impermeable o un plano de simetria, y Robin al
intercambio con un medio cuya resistencia no es despreciable, como la conveccion
sobre una superficie.

La celda siguiente resuelve la conduccion estacionaria en una pared plana con
las tres condiciones sobre la cara derecha, y compara cada resultado con su
solucion analitica, que es la verificacion que exige la etapa 5 del ciclo.

In [ ]:
ESPESOR = 0.20            # m
CONDUCTIVIDAD = 1.4       # W/(m K)
TEMPERATURA_IZQUIERDA = 90.0   # C
NODOS = 81


def conduccion_estacionaria(tipo, valor, nodos=NODOS):
    """Perfil de temperatura en una pared plana, en C, con tres fronteras."""
    dx = ESPESOR / (nodos - 1)
    matriz = np.zeros((nodos, nodos))
    lado = np.zeros(nodos)

    matriz[0, 0] = 1.0                       # Dirichlet en la cara izquierda
    lado[0] = TEMPERATURA_IZQUIERDA
    for i in range(1, nodos - 1):            # difusion pura en el interior
        matriz[i, i - 1], matriz[i, i], matriz[i, i + 1] = 1.0, -2.0, 1.0

    if tipo == "dirichlet":                  # temperatura impuesta
        matriz[-1, -1] = 1.0
        lado[-1] = valor
    elif tipo == "neumann":                  # flujo impuesto en W/m2
        matriz[-1, -2], matriz[-1, -1] = -1.0, 1.0
        lado[-1] = -valor * dx / CONDUCTIVIDAD
    elif tipo == "robin":                    # conveccion, valor = (h, T_inf)
        coeficiente, ambiente = valor
        matriz[-1, -2] = -CONDUCTIVIDAD / dx
        matriz[-1, -1] = CONDUCTIVIDAD / dx + coeficiente
        lado[-1] = coeficiente * ambiente
    else:
        raise ValueError("tipo de frontera no reconocido")

    return np.linspace(0.0, ESPESOR, nodos), np.linalg.solve(matriz, lado)


x_d, t_d = conduccion_estacionaria("dirichlet", 25.0)
x_n, t_n = conduccion_estacionaria("neumann", 210.0)
x_r, t_r = conduccion_estacionaria("robin", (12.0, 25.0))

# Verificacion contra la solucion analitica de cada caso.
analitica_d = TEMPERATURA_IZQUIERDA + (25.0 - TEMPERATURA_IZQUIERDA) * x_d / ESPESOR
analitica_n = TEMPERATURA_IZQUIERDA - 210.0 * x_n / CONDUCTIVIDAD
extremo_r = ((CONDUCTIVIDAD / ESPESOR * TEMPERATURA_IZQUIERDA + 12.0 * 25.0)
             / (CONDUCTIVIDAD / ESPESOR + 12.0))
analitica_r = TEMPERATURA_IZQUIERDA + (extremo_r - TEMPERATURA_IZQUIERDA) * x_r / ESPESOR

for nombre, numerica, exacta in (("Dirichlet", t_d, analitica_d),
                                 ("Neumann", t_n, analitica_n),
                                 ("Robin", t_r, analitica_r)):
    error = float(np.max(np.abs(numerica - exacta)))
    print(f"{nombre:<10s} error maximo frente a la solucion analitica "
          f"{error:.2e} C")
    assert error < 1e-9

figura, eje = plt.subplots()
eje.plot(x_d * 1000, t_d, color=PALETA["azul"], lw=1.9,
         label="Dirichlet, 25 C impuestos en la cara derecha")
eje.plot(x_n * 1000, t_n, color=PALETA["rojo"], lw=1.9, ls="--",
         label="Neumann, 210 W/m2 salientes")
eje.plot(x_r * 1000, t_r, color=PALETA["verde"], lw=1.9, ls="-.",
         label="Robin, conveccion con h = 12 W/(m2 K) y aire a 25 C")
eje.set_xlabel("Posicion dentro de la pared (mm)")
eje.set_ylabel("Temperatura (C)")
eje.legend(loc="lower left", fontsize=8.8)
eje.set_title("Los tres tipos de condicion de frontera sobre el mismo dominio")
plt.show()

### La formulacion sin solucion unica

El libro advierte que un problema de difusion con todas las fronteras aisladas y
sin termino fuente no tiene solucion unica, porque cualquier constante aditiva
satisface las mismas ecuaciones, y que el sintoma en el computador es una matriz
singular. Conviene provocar ese sintoma una vez, en un entorno controlado, para
reconocerlo cuando aparezca en un problema real.

In [ ]:
nodos = 21
matriz = np.zeros((nodos, nodos))
matriz[0, 0], matriz[0, 1] = 1.0, -1.0        # Neumann homogenea a la izquierda
matriz[-1, -2], matriz[-1, -1] = -1.0, 1.0    # Neumann homogenea a la derecha
for i in range(1, nodos - 1):
    matriz[i, i - 1], matriz[i, i], matriz[i, i + 1] = 1.0, -2.0, 1.0

rango = int(np.linalg.matrix_rank(matriz, tol=1e-8))
print(f"matriz de {nodos} por {nodos} con rango {rango}, deficiente en "
      f"{nodos - rango}")
try:
    np.linalg.solve(matriz, np.zeros(nodos))
    print("El solucionador no fallo, pero el resultado no es unico.")
except np.linalg.LinAlgError as error:
    print("np.linalg.LinAlgError ·", error)
print("\nCon todas las fronteras aisladas cualquier constante aditiva es "
      "solucion, de modo que falta una condicion que fije el nivel.")
assert rango == nodos - 1

## 6. Coherencia dimensional y el teorema Pi

La Seccion 1.5 del libro cierra con dos comprobaciones que cuestan muy poco y
atrapan errores caros. La primera es la coherencia dimensional, que intercepta el
error mas comun al programar una ecuacion tomada de la literatura, que consiste
en mezclar formulaciones por unidad de masa con formulaciones por unidad de
volumen. La segunda es el teorema Pi de Buckingham, que reduce el numero de
variables de un programa experimental.

In [ ]:
BASE = ("M", "L", "T", "K")


def dimension(**exponentes):
    """Vector de exponentes de una magnitud en las dimensiones fundamentales."""
    return np.array([exponentes.get(s, 0) for s in BASE], dtype=int)


def coherente(terminos):
    """Verdadero si todos los terminos comparten el mismo vector dimensional."""
    vectores = [sum(factores) for factores in terminos.values()]
    return all(np.array_equal(v, vectores[0]) for v in vectores)


ENERGIA = {
    "rho*cp*dT/dt": [dimension(M=1, L=-3), dimension(L=2, T=-2, K=-1),
                     dimension(K=1, T=-1)],
    "k*d2T/dx2":    [dimension(M=1, L=1, T=-3, K=-1), dimension(K=1, L=-2)],
    "h*(T-Tinf)/L": [dimension(M=1, T=-3, K=-1), dimension(K=1),
                     dimension(L=-1)],
}

for nombre, factores in ENERGIA.items():
    print(f"{nombre:<16s} {dict(zip(BASE, sum(factores)))}")
print(f"\nbalance de energia coherente · {coherente(ENERGIA)}")
assert coherente(ENERGIA), "los tres terminos deben ser una potencia por volumen"

### Ejemplo 1.3 del libro, semejanza de una bomba centrifuga

Una bomba centrifuga de rodete 200 mm se ensaya con agua a 20 grados girando a
1750 rpm, y en su punto de mejor rendimiento entrega 22 L/s contra una altura de
18 m. Se pide predecir el caudal, la altura y la potencia hidraulica del mismo
rodete girando a 2400 rpm.

Con seis magnitudes y una matriz dimensional de rango tres, el Teorema 1.2
anticipa tres grupos independientes. El Listado 1.6 del libro los obtiene con
algebra simbolica, resolviendo para cada magnitud no repetida el sistema lineal
que anula las dimensiones del producto correspondiente.

In [ ]:
# Filas M, L, T. Columnas gH, Q, N, D, rho, mu.
dimensional = sp.Matrix([[0, 0, 0, 0, 1, 1],
                         [2, 3, 0, 1, -3, -1],
                         [-2, -1, -1, 0, 0, -1]])
nombres = ["gH", "Q", "N", "D", "rho", "mu"]
repetidas, restantes = [4, 2, 3], [0, 1, 5]   # rho, N y D como base

rango_dimensional = dimensional.rank()
n_grupos = dimensional.shape[1] - rango_dimensional
base_repetidas = dimensional[:, repetidas]

print(f"rango de la matriz dimensional · {rango_dimensional}")
print(f"grupos adimensionales independientes · {n_grupos}\n")
for j in restantes:
    exponentes = base_repetidas.solve(-dimensional[:, j])
    factores = " ".join(
        f"{nombres[repetidas[i]]}^({sp.nsimplify(e)})" for i, e in enumerate(exponentes))
    print(f"  Pi de {nombres[j]:<4s} · {nombres[j]} {factores}")

verificar_libro("numero de grupos adimensionales", n_grupos, 3, 1e-9)

In [ ]:
DENSIDAD = 998.2          # kg/m3
VISCOSIDAD = 1.002e-3     # Pa s
DIAMETRO_RODETE = 0.200   # m

giro_uno = 1750 / 60.0    # 1/s
giro_dos = 2400 / 60.0    # 1/s
caudal_uno = 0.022        # m3/s
altura_uno = 18.0         # m

razon = giro_dos / giro_uno
caudal_dos = caudal_uno * razon                       # m3/s
altura_dos = altura_uno * razon**2                    # m
potencia_uno = DENSIDAD * GRAVEDAD * caudal_uno * altura_uno    # W
potencia_dos = DENSIDAD * GRAVEDAD * caudal_dos * altura_dos    # W

coef_caudal = caudal_uno / (giro_uno * DIAMETRO_RODETE**3)
coef_altura = GRAVEDAD * altura_uno / (giro_uno**2 * DIAMETRO_RODETE**2)
reynolds_uno = DENSIDAD * giro_uno * DIAMETRO_RODETE**2 / VISCOSIDAD
reynolds_dos = DENSIDAD * giro_dos * DIAMETRO_RODETE**2 / VISCOSIDAD

verificar_libro("caudal a 2400 rpm", caudal_dos * 1000, 30.17, 1e-3, "L/s")
verificar_libro("altura a 2400 rpm", altura_dos, 33.85, 1e-3, "m")
verificar_libro("potencia hidraulica a 1750 rpm", potencia_uno / 1000, 3.88, 1e-3, "kW")
verificar_libro("potencia hidraulica a 2400 rpm", potencia_dos / 1000, 10.00, 1e-3, "kW")
verificar_libro("razon de potencias", potencia_dos / potencia_uno, 2.579, 1e-3)
verificar_libro("razon de giros al cubo", razon**3, 2.579, 1e-3)
verificar_libro("coeficiente de caudal Pi1", coef_caudal, 0.0943, 1e-3)
verificar_libro("coeficiente de altura Pi2", coef_altura, 5.189, 1e-3)
verificar_libro("Reynolds del rodete a 1750 rpm", reynolds_uno, 1.16e6, 5e-3)
verificar_libro("Reynolds del rodete a 2400 rpm", reynolds_dos, 1.59e6, 5e-3)

coef_caudal_dos = caudal_dos / (giro_dos * DIAMETRO_RODETE**3)
coef_altura_dos = GRAVEDAD * altura_dos / (giro_dos**2 * DIAMETRO_RODETE**2)
print(f"\nPi1 se conserva · {coef_caudal:.6f} frente a {coef_caudal_dos:.6f}")
print(f"Pi2 se conserva · {coef_altura:.6f} frente a {coef_altura_dos:.6f}")
print("Pi3 no se conserva con el mismo fluido y el mismo rodete, de modo que la "
      "prediccion solo vale porque ambos Reynolds superan holgadamente 1e5.")

## 7. Ejercicios guiados

Seis celdas incompletas con la marca `# COMPLETE:`, un valor de partida
deliberadamente incorrecto y su verificacion inmediata.

### Ejercicio 1. Problema 1-13, otro canal trapezoidal

El Problema 1-13 pide el tirante normal de un canal con ancho de fondo 3.0 m,
talud 2.0, rugosidad 0.030 y pendiente 0.0005, para un caudal de 15 m3/s.

In [ ]:
# COMPLETE: use tirante_normal con los datos del Problema 1-13 y guarde el
# resultado en y_problema, en metros.
y_problema = 0.0                # valor de partida deliberadamente incorrecto

In [ ]:
comprobar("tirante normal del Problema 1-13", y_problema, 2.255623, 1e-5, "m")

### Ejercicio 2. El regimen del flujo en ese canal

In [ ]:
# COMPLETE: calcule el numero de Froude del canal del Problema 1-13 con el
# tirante que acaba de obtener, y deje en regimen_problema la cadena
# "subcritico" o "supercritico" segun corresponda.
froude_problema = 0.0           # valor de partida deliberadamente incorrecto
regimen_problema = "supercritico"   # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("numero de Froude del Problema 1-13", froude_problema, 0.238115, 1e-4)
if REVISAR:
    assert regimen_problema == "subcritico", "el flujo resulta subcritico"
    print("[correcto ]  regimen del flujo · subcritico")

### Ejercicio 3. Grados de libertad de un intercambiador

El Problema 1-12 describe un intercambiador de doble tubo con las temperaturas
de entrada y salida de los dos fluidos, los dos caudales masicos, los dos
calores especificos, el coeficiente global y el area, esto es diez magnitudes.
Las ecuaciones independientes son tres, que son los dos balances de energia y la
relacion de transferencia con la diferencia media logaritmica.

In [ ]:
# COMPLETE: calcule los grados de libertad del intercambiador antes de
# especificar nada, como la diferencia entre incognitas y ecuaciones
# independientes, y luego los que quedan si se especifican las dos temperaturas
# de entrada, los dos caudales, los dos calores especificos y el coeficiente
# global, esto es siete magnitudes.
gl_intercambiador = 0           # valor de partida deliberadamente incorrecto
gl_tras_especificar = 0         # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("grados de libertad del intercambiador", gl_intercambiador, 7, 1e-9)
dos = comprobar("grados tras siete especificaciones", gl_tras_especificar, 0, 1e-9)

### Ejercicio 4. Problema 1-14, clasificar cuatro fronteras

El Problema 1-14 pide clasificar como de Dirichlet, de Neumann o de Robin cuatro
situaciones. Un embalse que fija el nivel piezometrico en el borde de un
acuifero, una pared de horno aislada, la superficie de un producto enfriado por
aire forzado y la base de un lisimetro, que impone un flujo de drenaje medido.

In [ ]:
# COMPLETE: escriba en cada casilla la palabra dirichlet, neumann o robin.
fronteras = {
    "embalse en el borde del acuifero": "robin",     # valor de partida incorrecto
    "pared de horno aislada": "dirichlet",           # valor de partida incorrecto
    "producto enfriado por aire forzado": "neumann",  # valor de partida incorrecto
    "base de lisimetro con drenaje medido": "robin",  # valor de partida incorrecto
}

In [ ]:
ESPERADAS = {"embalse en el borde del acuifero": "dirichlet",
             "pared de horno aislada": "neumann",
             "producto enfriado por aire forzado": "robin",
             "base de lisimetro con drenaje medido": "neumann"}
aciertos = sum(fronteras.get(k) == v for k, v in ESPERADAS.items())
for situacion, esperada in ESPERADAS.items():
    dada = fronteras.get(situacion)
    print(f"  {situacion:<40s} respondio {str(dada):<10s}"
          f"{'' if dada == esperada else 'revisar'}")
comprobar("fronteras acertadas, sobre 4", aciertos, 4, 1e-9)

### Ejercicio 5. Coherencia dimensional de un balance con un error

El Problema 1-23 pide programar la verificacion del Listado 1.9 y aplicarla a
un balance de masa que incluya un termino escrito por error en base masica. El
balance de un tanque agitado, por unidad de volumen, tiene los terminos de
acumulacion, de entrada y de reaccion, todos con dimensiones de masa por volumen
y por tiempo, esto es M L^-3 T^-1.

In [ ]:
# COMPLETE: escriba el vector dimensional del termino que falta, que es un
# caudal masico dividido por el volumen, esto es masa sobre tiempo dividida por
# un volumen. Use la funcion dimension con los exponentes M, L y T.
BALANCE = {
    "dC/dt":        [dimension(M=1, L=-3, T=-1)],
    "Q/V*(Ce-C)":   [dimension(T=-1), dimension(M=1, L=-3)],
    "k*C":          [dimension(T=-1), dimension(M=1, L=-3)],
    "w/V":          [dimension(M=1)],   # vector de partida deliberadamente incorrecto
}
balance_coherente = coherente(BALANCE)

In [ ]:
comprobar("coherencia dimensional del balance", float(balance_coherente), 1.0, 1e-9)

### Ejercicio 6. Problema 1-18, los grupos de un agitador

El Problema 1-18 pide obtener los grupos adimensionales que gobiernan la
potencia de un agitador a partir de la potencia, el diametro del impulsor, la
velocidad de giro, la densidad y la viscosidad. La matriz dimensional, con filas
M, L y T y columnas P, D, N, rho y mu, es la que ya esta escrita.

In [ ]:
# COMPLETE: calcule el rango de la matriz dimensional del agitador y el numero
# de grupos adimensionales independientes que predice el Teorema 1.2.
matriz_agitador = sp.Matrix([[1, 0, 0, 1, 1],
                             [2, 1, 0, -3, -1],
                             [-3, 0, -1, 0, -1]])
rango_agitador = 0              # valor de partida deliberadamente incorrecto
grupos_agitador = 0             # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("rango de la matriz del agitador", rango_agitador, 3, 1e-9)
dos = comprobar("grupos adimensionales del agitador", grupos_agitador, 2, 1e-9)

## 8. Problemas del capitulo

- **1-11.** Escriba en la forma de la Ecuacion 1.4 el modelo dinamico de mezcla
  completa del Ejemplo 1.1, con su estado, entrada, perturbacion, parametro y
  salida. La celda de la seccion 1 de este cuaderno es la plantilla.
- **1-15.** El vaciado de un tanque por un orificio da una ecuacion cuyo miembro
  derecho es proporcional a la raiz del nivel. La hipotesis que falla en el
  Teorema 1.1 es la de Lipschitz, porque la raiz cuadrada tiene pendiente
  infinita en cero, de modo que al vaciarse el tanque el problema admite
  infinitas soluciones y el integrador entrega resultados que dependen de la
  tolerancia. El remedio es de modelacion, no de metodo numerico, y consiste en
  regularizar la expresion cerca del origen o en detener la integracion como un
  evento.
- **1-22.** Implemente una funcion que reciba los residuos de un modelo
  estacionario y devuelva sus grados de libertad, con la estrategia del
  Listado 1.4, y pruebela sobre el canal trapezoidal. Ya esta resuelto en la
  seccion 3 de este cuaderno, y la extension consiste en aplicarlo a su propio
  problema de investigacion.

La celda siguiente ilustra el Problema 1-15 sin resolverlo, para que se vea el
sintoma con los propios ojos.

In [ ]:
COEFICIENTE_VACIADO = 0.02      # m^(1/2)/s


def vaciado(t, x):
    """Nivel de un tanque que se vacia por un orificio, en m."""
    nivel, = x
    return [-COEFICIENTE_VACIADO * np.sqrt(max(nivel, 0.0))]


tiempo_teorico = 2.0 * np.sqrt(1.0) / COEFICIENTE_VACIADO     # s, hasta vaciarse
soluciones = {}
for tolerancia in (1e-4, 1e-8, 1e-12):
    resultado = solve_ivp(vaciado, (0.0, 1.2 * tiempo_teorico), [1.0],
                          rtol=tolerancia, atol=tolerancia * 1e-2,
                          dense_output=True)
    soluciones[tolerancia] = float(resultado.sol(tiempo_teorico)[0])

print(f"tiempo teorico de vaciado · {tiempo_teorico:.1f} s")
for tolerancia, nivel in soluciones.items():
    print(f"  con tolerancia {tolerancia:.0e} el nivel al tiempo teorico vale "
          f"{nivel:.3e} m")
print("\nEl nivel al que llega el integrador depende de la tolerancia, que es "
      "el sintoma de la falta de unicidad que anuncia el Teorema 1.1.")

## Cierre

### Lista de comprobacion

Marque cada punto solo si puede hacerlo sin mirar el cuaderno.

- Escribir un modelo dinamico en la forma canonica y decir de cada magnitud si es estado, entrada, perturbacion o parametro.
- Levantar un registro de supuestos en el que cada fila lleve una prueba prevista.
- Contar los grados de libertad de una formulacion y detectar ecuaciones redundantes antes de programar.
- Resolver el tirante normal de un canal y clasificar el regimen con el numero de Froude.
- Reconocer los tres tipos de condicion de frontera y el sintoma de una formulacion sin solucion unica.
- Comprobar la coherencia dimensional de un balance y obtener sus grupos adimensionales con SymPy.

### Que revisar si algo no salio

- Si el tirante normal no coincide, revise que el radio hidraulico sea el area sobre el perimetro y que la pendiente entre bajo raiz cuadrada, no elevada a un medio de la rugosidad.
- Si los grados de libertad salen distintos de cero, cuente de nuevo las relaciones geometricas, que tambien son ecuaciones.
- Si el sistema de la pared no converge, verifique que la fila de la frontera de Robin lleva la conductividad sobre el paso de malla y no la conductividad sola.
- Para la teoria, relea la Seccion 1.4 del libro, las Definiciones 1.7, 1.8 y 1.9, los Ejemplos 1.2 y 1.3, la Tabla 1.3 y el Teorema 1.1.

### Declaracion del uso de asistentes de programacion

Si empleo un asistente basado en modelos de lenguaje para resolver alguna celda, declarelo en la entrega, indique en cual y describa que prueba aplico para convencerse de que el codigo es correcto. La regla de la asignatura es que el estudiante responde por el resultado que firma, con independencia de quien escriba las lineas.